In [1]:
# Librerías necesarias
import pandas as pd
from scipy.stats import ttest_rel
from sklearn.model_selection import train_test_split, RepeatedKFold
import numpy as np
import pandas as pd
import os
import pickle
import warnings

from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed
from sklearn.model_selection import train_test_split, RepeatedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import log_loss
from sklearn import tree

import matplotlib.pyplot as plt
import re
from scores import *

In [2]:
# file_path = "./resultados/resultados_media_binarios7_2025_06_05_16_17_46.csv"
# file_path_todo = "./resultados/resultados_binarios7_2025_06_05_16_17_46.csv"

file_path1 = "./resultados_finales/resultados_media_binarios8_2026_02_20_09_56_38.csv"
file_path_todo1 = "./resultados_finales/resultados_binarios8_2026_02_20_09_56_38.csv"

file_path2 = "./resultados_finales/resultados_media_binarios8_2026_02_20_09_58_12.csv"
file_path_todo2 = "./resultados_finales/resultados_binarios8_2026_02_20_09_58_12.csv"

file_path3 = "./resultados_finales/resultados_media_binarios8_2026_02_22_19_51_13.csv"
file_path_todo3 = "./resultados_finales/resultados_binarios8_2026_02_22_19_51_13.csv"

file_path4 = "./resultados_finales/resultados_media_binarios8_2026_02_24_09_17_14.csv"
file_path_todo4 = "./resultados_finales/resultados_binarios8_2026_02_24_09_17_14.csv"

file_path5 = "./resultados_finales/resultados_media_binarios8_2026_02_21_13_06_01.csv"
file_path_todo5 = "./resultados_finales/resultados_binarios8_2026_02_21_13_06_01.csv"

file_path6 = "./resultados_finales/resultados_media_binarios8_2026_02_25_16_20_35.csv"
file_path_todo6 = "./resultados_finales/resultados_binarios8_2026_02_25_16_20_35.csv"

df1 = pd.read_csv(file_path1)
df2 = pd.read_csv(file_path2)
df3 = pd.read_csv(file_path3)
df4 = pd.read_csv(file_path4)
df5 = pd.read_csv(file_path5)
df6 = pd.read_csv(file_path6)
df = pd.concat([df1, df2, df3, df4, df5, df6]).reset_index(drop=True)
df_orig = df.copy()

df1 = pd.read_csv(file_path_todo1)
df2 = pd.read_csv(file_path_todo2)
df3 = pd.read_csv(file_path_todo3)
df4 = pd.read_csv(file_path_todo4)
df5 = pd.read_csv(file_path_todo5)
df6 = pd.read_csv(file_path_todo6)
df = pd.concat([df1, df2, df3, df4, df5, df6]).reset_index(drop=True)
df_total = df.copy()

# Tabla Medias

In [3]:
# Cargar el archivo CSV
# file_path = "./resultados/resultados_media_binarios4_2025_06_02_14_42_34.csv"
#file_path = "./resultados/resultados_media_binarios5_2025_06_04_10_07_21.csv"
# file_path = "./resultados/resultados_media_binarios7_2025_06_05_16_17_46.csv"

df = df_orig.copy()
# Eliminar ".csv" de los nombres de dataset
df["dataset"] = df["dataset"].str.replace(".csv", "", regex=False)

# Redondear y formatear columnas según lo solicitado
df["i_5CV_logloss"] = df["i_5CV_logloss"].round(3)
df["p_5CV_logloss"] = df["p_5CV_logloss"].round(3)
df["i_TST_logloss"] = df["i_TST_logloss"].round(3)
df["p_TST_logloss"] = df["p_TST_logloss"].round(3)
df["i_geo_complexity"] = df["i_geo_complexity"].round(3)
df["p_geo_complexity"] = df["p_geo_complexity"].round(3)
df["i_tree_nfs"] = df["i_tree_nfs"].round(1)
df["p_tree_nfs"] = df["p_tree_nfs"].round(1)
df["i_H_active"] = df["i_H_active"].round(1)
df["p_H_active"] = df["p_H_active"].round(1)
df["i_C_inconsistentes"] = df["i_C_inconsistentes"].round(1)
df["p_C_inconsistentes"] = df["p_C_inconsistentes"].round(1)
df["i_k_promedio_activas"] = df["i_k_promedio_activas"].round(1)
df["p_k_promedio_activas"] = df["p_k_promedio_activas"].round(1)
df["nrows"] = df["nrows"].astype(int)
df["ncols"] = df["ncols"].astype(int)

# Ordenar por número de columnas
df_ordenado = df.sort_values(by="ncols")

# Construir un DataFrame limpio con las columnas ya procesadas y formateadas
df_final = df_ordenado[[
    "dataset", "nrows", "ncols",
    # "i_5CV_logloss", "p_5CV_logloss",
    "i_TST_logloss", "p_TST_logloss",
    "i_geo_complexity", "p_geo_complexity",
    "i_tree_nfs", "p_tree_nfs",
    "i_H_active", "p_H_active",
    "i_C_inconsistentes", "p_C_inconsistentes",
    "i_k_promedio_activas", "p_k_promedio_activas"
]].copy()

df_final = df_final.reset_index(drop=True)
df_final

,dataset,nrows,ncols,i_TST_logloss,p_TST_logloss,i_geo_complexity,p_geo_complexity,i_tree_nfs,p_tree_nfs,i_H_active,p_H_active,i_C_inconsistentes,p_C_inconsistentes,i_k_promedio_activas,p_k_promedio_activas
0,Titanic,2201,4,0.484,0.484,18.720,17.920,3.0,3.0,9.4,9.0,2.9,2.9,2.2,2.2
1,Long,4477,20,0.002,0.002,1.120,1.338,3.7,3.5,1.0,1.2,0.0,0.0,2.7,2.6
2,compas,5278,20,0.603,0.604,2.809,4.020,7.2,5.0,2.2,3.3,1.9,2.0,5.3,3.7
3,twonorm,7400,21,0.062,0.062,6.876,17.676,19.9,19.9,4.4,11.6,0.9,2.1,14.9,14.9
4,ringnorm,7400,21,0.188,0.207,22.151,18.657,13.6,11.3,13.0,11.8,13.6,11.3,10.2,8.4
5,kc1,2109,22,0.355,0.354,9.656,9.615,8.0,5.6,1.4,4.5,0.8,2.6,5.3,3.8
6,Contaminant,2400,31,0.223,0.206,10.092,16.160,20.4,20.2,5.9,9.6,12.4,14.5,15.3,15.1
7,churn,5000,34,0.186,0.174,5.642,9.240,17.2,12.0,3.9,6.8,11.0,9.5,12.8,8.9
8,Satellite,5100,37,0.024,0.026,2.507,6.188,10.3,7.2,2.0,1.9,1.2,0.4,7.7,5.1
9,mc1,9466,39,0.029,0.030,9.592,3.951,15.2,6.2,3.9,3.4,8.8,3.3,11.2,4.6


In [11]:
def strip_leading_zero_decimal(s: str) -> str:
    # s viene ya como string tipo "0.123" o "-0.123"
    if s.startswith("0."):
        return s[1:]          # "0.123" -> ".123"
    if s.startswith("-0."):
        return "-" + s[2:]    # "-0.123" -> "-.123"
    return s


# Crear una copia para formatear como strings
df_tex = df_final.copy()

# Formatear columnas numéricas a strings con el formato deseado
df_tex["nrows"] = df_tex["nrows"].astype(str)
df_tex["ncols"] = df_tex["ncols"].astype(str)
# df_tex["i_5CV_logloss"] = df_tex["i_5CV_logloss"].map("{:.3f}".format)
# df_tex["p_5CV_logloss"] = df_tex["p_5CV_logloss"].map("{:.3f}".format)
# df_tex["i_TST_logloss"] = df_tex["i_TST_logloss"].map("{:.3f}".format)
# df_tex["p_TST_logloss"] = df_tex["p_TST_logloss"].map("{:.3f}".format)

# Formatear a 3 decimales y quitar el 0 inicial
df_tex["i_TST_logloss"] = df_final["i_TST_logloss"].map("{:.3f}".format).map(strip_leading_zero_decimal)
df_tex["p_TST_logloss"] = df_final["p_TST_logloss"].map("{:.3f}".format).map(strip_leading_zero_decimal)

df_tex["i_geo_complexity"] = df_tex["i_geo_complexity"].map("{:.1f}".format)
df_tex["p_geo_complexity"] = df_tex["p_geo_complexity"].map("{:.1f}".format)
df_tex["i_tree_nfs"] = df_tex["i_tree_nfs"].map("{:.1f}".format)
df_tex["p_tree_nfs"] = df_tex["p_tree_nfs"].map("{:.1f}".format)
df_tex["i_H_active"] = df_tex["i_H_active"].map("{:.1f}".format)
df_tex["p_H_active"] = df_tex["p_H_active"].map("{:.1f}".format)
df_tex["i_C_inconsistentes"] = df_tex["i_C_inconsistentes"].map("{:.1f}".format)
df_tex["p_C_inconsistentes"] = df_tex["p_C_inconsistentes"].map("{:.1f}".format)
df_tex["i_k_promedio_activas"] = df_tex["i_k_promedio_activas"].map("{:.1f}".format)
df_tex["p_k_promedio_activas"] = df_tex["p_k_promedio_activas"].map("{:.1f}".format)
# Ahora imprime la tabla en LaTeX con los valores ya formateados
print(df_tex.to_latex(index=False, escape=False))


\begin{tabular}{lllllllllllllll}
\toprule
dataset & nrows & ncols & i_TST_logloss & p_TST_logloss & i_geo_complexity & p_geo_complexity & i_tree_nfs & p_tree_nfs & i_H_active & p_H_active & i_C_inconsistentes & p_C_inconsistentes & i_k_promedio_activas & p_k_promedio_activas \\
\midrule
Titanic & 2201 & 4 & .484 & .484 & 18.7 & 17.9 & 3.0 & 3.0 & 9.4 & 9.0 & 2.9 & 2.9 & 2.2 & 2.2 \\
Long & 4477 & 20 & .002 & .002 & 1.1 & 1.3 & 3.7 & 3.5 & 1.0 & 1.2 & 0.0 & 0.0 & 2.7 & 2.6 \\
compas & 5278 & 20 & .603 & .604 & 2.8 & 4.0 & 7.2 & 5.0 & 2.2 & 3.3 & 1.9 & 2.0 & 5.3 & 3.7 \\
twonorm & 7400 & 21 & .062 & .062 & 6.9 & 17.7 & 19.9 & 19.9 & 4.4 & 11.6 & 0.9 & 2.1 & 14.9 & 14.9 \\
ringnorm & 7400 & 21 & .188 & .207 & 22.2 & 18.7 & 13.6 & 11.3 & 13.0 & 11.8 & 13.6 & 11.3 & 10.2 & 8.4 \\
kc1 & 2109 & 22 & .355 & .354 & 9.7 & 9.6 & 8.0 & 5.6 & 1.4 & 4.5 & 0.8 & 2.6 & 5.3 & 3.8 \\
Contaminant & 2400 & 31 & .223 & .206 & 10.1 & 16.2 & 20.4 & 20.2 & 5.9 & 9.6 & 12.4 & 14.5 & 15.3 & 15.1 \\
churn & 5000

# Tabla Diferencias

In [5]:

df = df_orig.copy()

# Definir los pares a comparar
pares = [
    ("i_5CV_logloss", "p_5CV_logloss"),
    ("i_TST_logloss", "p_TST_logloss"),
    ("i_geo_complexity", "p_geo_complexity"),
    ("i_tree_nfs", "p_tree_nfs"),
    ("i_H_active", "p_H_active"),
    ("i_C_inconsistentes", "p_C_inconsistentes"),
    ("i_k_promedio_activas", "p_k_promedio_activas"),
]


# Calcular métricas y almacenar resultados
resultados = []

for col_i, col_p in pares:
    menor = (df_orig[col_i].round(3) < df[col_p].round(3)).mean() * 100
    igual = (df[col_i].round(3) == df[col_p].round(3)).mean() * 100
    mayor = (df[col_i].round(3) > df[col_p].round(3)).mean() * 100
    stat, pval = ttest_rel(df[col_i].round(3), df[col_p].round(3))
    significativo = pval < 0.05
    resultados.append({
        "Par": f"{col_i} vs {col_p}",
        "Menor(%)": round(menor, 2),
        "Igual(%)": round(igual, 2),
        "Mayor(%)": round(mayor, 2),
        "p-value": round(pval, 4),
        "Significativo (p<0.05)": significativo
    })

# Convertir a DataFrame y mostrar
df_resultados = pd.DataFrame(resultados)
print(df_resultados)

                                            Par  Menor(%)  Igual(%)  Mayor(%)  \
0                i_5CV_logloss vs p_5CV_logloss     34.48     24.14     41.38   
1                i_TST_logloss vs p_TST_logloss     37.93     24.14     37.93   
2          i_geo_complexity vs p_geo_complexity     82.76      0.00     17.24   
3                      i_tree_nfs vs p_tree_nfs      6.90      6.90     86.21   
4                      i_H_active vs p_H_active     79.31      3.45     17.24   
5      i_C_inconsistentes vs p_C_inconsistentes     58.62      3.45     37.93   
6  i_k_promedio_activas vs p_k_promedio_activas      6.90      3.45     89.66   

   p-value  Significativo (p<0.05)  
0   0.4255                   False  
1   0.3139                   False  
2   0.0020                    True  
3   0.0051                    True  
4   0.0000                    True  
5   0.7512                   False  
6   0.0037                    True  


In [6]:
# Crear una copia para formatear como strings
df_tex = df_resultados.copy()

# Formatear columnas numéricas a strings con el formato deseado
df_tex["Menor(%)"] = df_tex["Menor(%)"].map("{:.2f}".format)
df_tex["Igual(%)"] = df_tex["Igual(%)"].map("{:.2f}".format)
df_tex["Mayor(%)"] = df_tex["Mayor(%)"].map("{:.2f}".format)
df_tex["p-value"] = df_tex["p-value"].map("{:.4}".format)

# Ahora imprime la tabla en LaTeX con los valores ya formateados
print(df_tex.to_latex(index=False, escape=False))

\begin{tabular}{lllllr}
\toprule
Par & Menor(%) & Igual(%) & Mayor(%) & p-value & Significativo (p<0.05) \\
\midrule
i_5CV_logloss vs p_5CV_logloss & 34.48 & 24.14 & 41.38 & 0.4255 & False \\
i_TST_logloss vs p_TST_logloss & 37.93 & 24.14 & 37.93 & 0.3139 & False \\
i_geo_complexity vs p_geo_complexity & 82.76 & 0.00 & 17.24 & 0.002 & True \\
i_tree_nfs vs p_tree_nfs & 6.90 & 6.90 & 86.21 & 0.0051 & True \\
i_H_active vs p_H_active & 79.31 & 3.45 & 17.24 & 0.0 & True \\
i_C_inconsistentes vs p_C_inconsistentes & 58.62 & 3.45 & 37.93 & 0.7512 & False \\
i_k_promedio_activas vs p_k_promedio_activas & 6.90 & 3.45 & 89.66 & 0.0037 & True \\
\bottomrule
\end{tabular}



In [7]:
df = df_orig.copy()
df[['dataset', 'run', 'seed', 'nrows', 'ncols', 'i_H_total',
       'i_H_active', 'p_H_total',
       'p_H_active']]

,dataset,run,seed,nrows,ncols,i_H_total,i_H_active,p_H_total,p_H_active
0,Long.csv,12.0000,14808.0000,4477.0,20.0,1.04,1.04,1.24,1.24
1,Titanic.csv,12.0000,14808.0000,2201.0,4.0,9.68,9.36,9.52,8.96
2,compas.csv,12.0000,14808.0000,5278.0,20.0,2.20,2.20,3.52,3.32
3,twonorm.csv,12.0000,14808.0000,7400.0,21.0,4.44,4.44,11.76,11.60
4,SpeedDating.csv,12.0000,14808.0000,8378.0,501.0,3.16,3.16,9.80,9.36
5,clean2.csv,12.0000,14808.0000,6598.0,169.0,1.00,1.00,9.28,9.08
6,jasmine.csv,12.0000,14808.0000,2984.0,281.0,2.04,2.04,7.88,7.28
7,madeline.csv,12.0000,14808.0000,3140.0,260.0,9.48,9.00,9.72,7.96
8,philippine.csv,12.0000,14808.0000,5832.0,309.0,2.00,2.00,10.32,10.20
9,scene.csv,12.0000,14808.0000,2407.0,305.0,2.96,2.96,10.20,10.16


# Grafico de Pareto COIL2000

In [8]:
# # 1. Load the CSV
# df1 = pd.read_csv(file_path_todo1)
# df2 = pd.read_csv(file_path_todo2)
# df3 = pd.read_csv(file_path_todo3)
# df = pd.concat([df1, df2, df3]).reset_index(drop=True)

# # 2. Filter Coil2000 rows (if needed; skip if file is only Coil2000)
# coil = df[df["dataset"] == "coil2000.csv"].reset_index(drop=True)

# # 3. Select only the i_ columns plus the run index
# i_cols = [
#     "run",
#     "i_5CV_logloss",
#     "i_TST_logloss",
#     "i_geo_complexity",
#     "i_tree_nfs"
# ]
# df_i = coil[i_cols].copy()

# # 4. Compute Pareto-frontier among i_ solutions (minimize both metrics)
# pareto_mask = np.ones(len(df_i), dtype=bool)
# for i in range(len(df_i)):
#     if not pareto_mask[i]:
#         continue
#     dominated = (
#         (df_i["i_5CV_logloss"] <= df_i.loc[i, "i_5CV_logloss"]) &
#         (df_i["i_geo_complexity"] <= df_i.loc[i, "i_geo_complexity"])
#     )
#     dominated.iloc[i] = False
#     if dominated.any():
#         pareto_mask[i] = False

# pareto_i = df_i[pareto_mask].reset_index(drop=True)

# # 5. Plot all i_ solutions and highlight Pareto-optimal points
# plt.figure(figsize=(8, 6))
# plt.scatter(
#     df_i["i_5CV_logloss"],
#     df_i["i_geo_complexity"],
#     marker='x',
#     color='blue',
#     alpha=0.7,
#     label='All i_ Solutions'
# )
# plt.scatter(
#     pareto_i["i_5CV_logloss"],
#     pareto_i["i_geo_complexity"],
#     facecolors='none',
#     edgecolors='red',
#     s=80,
#     linewidths=1.5,
#     label='Pareto-optimal i_'
# )
# x_label = rf'$5CV_i$ Logloss'
# y_label = rf'$i(T;\tau)_i$'
# plt.xlabel(x_label)
# plt.ylabel(y_label)
# # plt.xlabel("i_5CV_logloss")
# # plt.ylabel("i_geo_complexity")
# plt.title("Coil2000: Interpretability Solutions with Pareto Optimals Highlighted")
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.savefig(f'Pareto_interpretabily_coil2000.png', dpi=600)
# plt.show()


In [9]:
# # 1. Load the CSV
# df1 = pd.read_csv(file_path_todo1)
# df2 = pd.read_csv(file_path_todo2)
# df3 = pd.read_csv(file_path_todo3)
# df = pd.concat([df1, df2, df3]).reset_index(drop=True)

# # 2. Filter Coil2000 rows (if needed; skip if file is only Coil2000)
# coil = df[df["dataset"] == "coil2000.csv"].reset_index(drop=True)

# # 3. Select only the p_ columns plus the run index
# p_cols = [
#     "run",
#     "p_5CV_logloss",
#     "p_TST_logloss",
#     "p_geo_complexity",
#     "p_tree_nfs"
# ]
# df_p = coil[p_cols].copy()

# # 4. Compute Pareto-frontier among p_ solutions (minimize both metrics)
# pareto_mask_p = np.ones(len(df_p), dtype=bool)
# for i in range(len(df_p)):
#     if not pareto_mask_p[i]:
#         continue
#     dominated = (
#         (df_p["p_5CV_logloss"] <= df_p.loc[i, "p_5CV_logloss"]) &
#         (df_p["p_geo_complexity"] <= df_p.loc[i, "p_geo_complexity"])
#     )
#     dominated.iloc[i] = False
#     if dominated.any():
#         pareto_mask_p[i] = False

# pareto_p = df_p[pareto_mask_p].reset_index(drop=True)

# # 5. Plot all p_ solutions and highlight Pareto-optimal points
# plt.figure(figsize=(8, 6))
# plt.scatter(
#     df_p["p_5CV_logloss"],
#     df_p["p_geo_complexity"],
#     marker='x',
#     color='green',
#     alpha=0.7,
#     label='All p_ Solutions'
# )
# plt.scatter(
#     pareto_p["p_5CV_logloss"],
#     pareto_p["p_geo_complexity"],
#     facecolors='none',
#     edgecolors='red',
#     s=80,
#     linewidths=1.5,
#     label='Pareto-optimal p_'
# )
# x_label = rf'$5CV_p$ Logloss'
# y_label = rf'$i(T;\tau)_p$'
# plt.xlabel(x_label)
# plt.ylabel(y_label)
# plt.title("Coil2000: Parsimonious Solutions with Pareto Optimals Highlighted")
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.savefig(f'Pareto_parsimonious_coil2000.png', dpi=600)
# plt.show()

In [10]:
# # 1. Load the CSV (replace with the actual filepath if needed)
# df1 = pd.read_csv(file_path_todo1)
# df2 = pd.read_csv(file_path_todo2)
# df3 = pd.read_csv(file_path_todo3)
# df = pd.concat([df1, df2, df3]).reset_index(drop=True)


# # 2. If the file contains multiple datasets, filter for Coil2000 rows
# #    (skip this step if the file only has Coil2000)
# coil = df[df["dataset"] == "coil2000.csv"].reset_index(drop=True)

# # 3. Extract the i_ and p_ subsets
# i_cols = ["run", "i_5CV_logloss", "i_geo_complexity"]
# p_cols = ["run", "p_5CV_logloss", "p_geo_complexity"]
# df_i = coil[i_cols].copy()
# df_p = coil[p_cols].copy()

# # 4. Compute Pareto-frontier among i_ solutions (minimize both metrics)
# pareto_mask_i = np.ones(len(df_i), dtype=bool)
# for i in range(len(df_i)):
#     if not pareto_mask_i[i]:
#         continue
#     dominated_i = (
#         (df_i["i_5CV_logloss"] <= df_i.loc[i, "i_5CV_logloss"]) &
#         (df_i["i_geo_complexity"] <= df_i.loc[i, "i_geo_complexity"])
#     )
#     dominated_i.iloc[i] = False
#     if dominated_i.any():
#         pareto_mask_i[i] = False
# pareto_i = df_i[pareto_mask_i].reset_index(drop=True)

# # 5. Compute Pareto-frontier among p_ solutions
# pareto_mask_p = np.ones(len(df_p), dtype=bool)
# for i in range(len(df_p)):
#     if not pareto_mask_p[i]:
#         continue
#     dominated_p = (
#         (df_p["p_5CV_logloss"] <= df_p.loc[i, "p_5CV_logloss"]) &
#         (df_p["p_geo_complexity"] <= df_p.loc[i, "p_geo_complexity"])
#     )
#     dominated_p.iloc[i] = False
#     if dominated_p.any():
#         pareto_mask_p[i] = False
# pareto_p = df_p[pareto_mask_p].reset_index(drop=True)

# # 6. Plot all points and highlight Pareto-optimal ones
# plt.figure(figsize=(8, 6))

# # i_ solutions: blue crosses
# plt.scatter(
#     df_i["i_5CV_logloss"],
#     df_i["i_geo_complexity"],
#     marker='x',
#     color='blue',
#     alpha=0.7,
#     label='All interpretability solutions'
# )
# # i_ Pareto: blue circles
# plt.scatter(
#     pareto_i["i_5CV_logloss"],
#     pareto_i["i_geo_complexity"],
#     facecolors='none',
#     edgecolors='blue',
#     s=80,
#     linewidths=1.5,
#     label='Interpretability Pareto frontier'
# )

# # p_ solutions: red crosses
# plt.scatter(
#     df_p["p_5CV_logloss"],
#     df_p["p_geo_complexity"],
#     marker='x',
#     color='red',
#     alpha=0.7,
#     label='All parsimonious solutions'
# )
# # p_ Pareto: red circles
# plt.scatter(
#     pareto_p["p_5CV_logloss"],
#     pareto_p["p_geo_complexity"],
#     facecolors='none',
#     edgecolors='red',
#     s=80,
#     linewidths=1.5,
#     label='Parsimonious Pareto frontier'
# )
# x_label = rf'$5CV$ Logloss'
# y_label = rf'$i(T;\tau)$'
# plt.xlabel(x_label)
# plt.ylabel(y_label)
# plt.title("Coil2000: IDT (blue) vs. PARSIMONIOUS (red) Solutions with Pareto Frontiers")
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.savefig(f'Pareto_idt_vs_parsimonious_coil2000.png', dpi=600)
# plt.show()
